# MemAgent with GraalPy Sandbox — Local Code Execution (No Cloud Required)

This notebook demonstrates how to use **MemAgent** with the **GraalPy** sandbox provider for **local, offline code execution** — no API keys, no cloud services, no billing.

## What is GraalPy?

GraalPy is a Python 3 runtime built on **GraalVM** by Oracle. When used as a sandbox provider, it gives MemAgent the ability to execute code locally with optional JVM-based security isolation.

### Two Execution Modes

| Mode | Security | Setup | Best For |
|------|----------|-------|----------|
| `subprocess` (default) | OS process isolation | Just install GraalPy | Development, testing |
| `java_wrapper` | JVM SandboxPolicy (UNTRUSTED) | JVM + wrapper JAR | Production, multi-tenant |

## What You'll Learn

1. How to set up GraalPy on your machine
2. How to configure MemAgent with GraalPy sandbox (subprocess mode)
3. How to use the java_wrapper mode for stronger isolation
4. When to choose GraalPy over cloud providers

---
## Step 1: Install GraalPy

GraalPy is a system dependency (not a pip package). Install it for your platform:

**macOS (Homebrew):**
```bash
brew install graalpy
```

**Linux:**
```bash
# Download from https://github.com/oracle/graalpython/releases
# Or use SDKMAN:
sdk install java 25.0.1-graal
```

**Verify installation:**
```bash
graalpy --version
```

If `graalpy` is not on your PATH, you can pass the full path when configuring the provider.

In [ ]:
# Install memorizz and OpenAI (GraalPy itself has no pip dependency)
%pip install -qU memorizz
%pip install -qU openai

print("Packages installed successfully!")

---
## Step 2: Verify GraalPy is Available

Let's check if GraalPy is installed and accessible.

In [ ]:
import shutil

graalpy_path = shutil.which("graalpy")
if graalpy_path:
    print(f"GraalPy found at: {graalpy_path}")
else:
    print("GraalPy not found on PATH.")
    print("Install it from: https://www.graalvm.org/python/")
    print("Or specify the full path when creating the provider.")

---
## Step 3: Configure API Key

GraalPy itself needs no API key (it's local!). You only need an OpenAI key for the LLM.

In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

---
## Step 4: Create a MemAgent with GraalPy Sandbox

### Subprocess Mode (Default)

The simplest setup — code runs via `graalpy -c "<code>"` in a subprocess.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True
)

In [ ]:
from memorizz.memagent.core import MemAgent
from memorizz.llms.llm_factory import create_llm_provider

llm = create_llm_provider({
    "provider": "openai",
    "model": "gpt-4o",
    "api_key": os.getenv("OPENAI_API_KEY"),
})

# Subprocess mode (default) — simplest setup
agent = MemAgent(
    model=llm,
    sandbox_provider="graalpy",
    instruction=(
        "You are a helpful coding assistant. "
        "Use the execute_code tool to run Python code when needed. "
        "Always explain your reasoning and show the output."
    ),
)

print(f"Sandbox provider: {agent.get_sandbox_provider_name()}")
print(f"Has sandbox: {agent.has_sandbox()}")

### Java Wrapper Mode (Advanced)

For production or multi-tenant environments, the `java_wrapper` mode uses GraalVM's `SandboxPolicy.UNTRUSTED` to fully restrict file system, network, and CPU access.

This requires a JVM and a wrapper JAR on your system.

In [ ]:
# Java wrapper mode (uncomment to use)
# Requires: JVM installed + wrapper JAR built

# agent_secure = MemAgent(
#     model=llm,
#     sandbox_provider={
#         "provider": "graalpy",
#         "mode": "java_wrapper",
#         "java_wrapper_jar": "/path/to/graalpy-sandbox.jar",
#         "sandbox_policy": "UNTRUSTED",  # Maximum isolation
#     },
#     instruction="You are a secure coding assistant.",
# )

---
## Step 5: Code Generation and Execution

### Example 1: Basic Computation

In [ ]:
response = agent.run(
    "Calculate the factorial of 20 and tell me how many digits it has."
)
print(f"\nAgent: {response}")

### Example 2: String Manipulation

In [ ]:
response = agent.run(
    "Write code to analyze the sentence: "
    "'The quick brown fox jumps over the lazy dog'. "
    "Count each letter's frequency and check if it's a pangram."
)
print(f"\nAgent: {response}")

### Example 3: Math Problem Solving

In [ ]:
response = agent.run(
    "Solve this: A farmer has chickens and cows. "
    "Together they have 30 heads and 86 legs. "
    "How many chickens and how many cows does the farmer have? "
    "Write code to solve this system of equations."
)
print(f"\nAgent: {response}")

---
## Step 6: Direct Code Execution

Test the sandbox directly without the LLM.

In [ ]:
import json

# Execute code directly
result_json = agent.execute_code("""
import sys
print(f"Python implementation: {sys.implementation.name}")
print(f"Version: {sys.version}")

# Quick computation
result = sum(range(1, 101))
print(f"Sum of 1 to 100: {result}")
""")

result = json.loads(result_json)
print(f"Success: {result['success']}")
for line in result['stdout']:
    print(f"  {line}")

---
## Key Takeaways

1. **No cloud dependency** — GraalPy runs entirely on your machine. No API keys, no network, no billing.
2. **Two security modes** — `subprocess` for development, `java_wrapper` for production.
3. **Same MemAgent API** — Switching between GraalPy, E2B, and Daytona requires no code changes beyond the provider config.
4. **Python only** — GraalPy currently supports Python execution only (E2B and Daytona support multiple languages).

## When to Use GraalPy

| Scenario | GraalPy? |
|----------|----------|
| Air-gapped / offline environments | Yes |
| Development and testing | Yes |
| Cost-sensitive (no cloud budget) | Yes |
| Production multi-tenant (with java_wrapper) | Yes |
| Need GPU support | No (use Daytona) |
| Need multi-language execution | No (use E2B) |